##### Build a RESTful API with Flask and SQLite – A Simple Task Manager

You are tasked with building a RESTful API using Flask and SQLite. The API will manage a task list, allowing users to create, retrieve, update, and delete tasks. This is a simple CRUD (Create, Read, Update, Delete) operation setup that will help you practice building APIs and working with a database.

Task Overview:
You need to implement the following API functionality:

Create a task, Retrieve all tasks, Retrieve a single task by ID, Update a task, and Delete a task.

Each task will have the following fields:

* id: Unique identifier (integer, auto-incremented by SQLite)
* title: A brief description of the task (string)
* description: A more detailed explanation of the task (string)
* done: Boolean field to represent whether the task is completed (True/False)

In [4]:
from flask import Flask, request, jsonify
import sqlite3
from werkzeug.serving import run_simple

app = Flask(__name__)
DATABASE = 'tasks.db'

# ------------------- Database Connection -------------------
def get_db():
    return sqlite3.connect(DATABASE)

# ------------------- Create Table -------------------
def init_db():
    db = get_db()
    db.execute('''
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            description TEXT,
            done BOOLEAN NOT NULL CHECK (done IN (0, 1))
        )
    ''')
    db.commit()
    db.close()

init_db()

# ------------------- Routes -------------------

@app.route('/tasks', methods=['GET'])
def get_tasks():
    db = get_db()
    db.row_factory = sqlite3.Row
    tasks = db.execute('SELECT * FROM tasks').fetchall()
    db.close()
    return jsonify([dict(task) for task in tasks])

@app.route('/tasks/<int:task_id>', methods=['GET'])
def get_task(task_id):
    db = get_db()
    db.row_factory = sqlite3.Row
    task = db.execute('SELECT * FROM tasks WHERE id = ?', (task_id,)).fetchone()
    db.close()
    if task is None:
        return jsonify({'error': 'Task not found'}), 404
    return jsonify(dict(task))

@app.route('/tasks', methods=['POST'])
def create_task():
    data = request.get_json()
    title = data.get('title')
    description = data.get('description', '')
    done = data.get('done', False)


    db = get_db()
    cursor = db.cursor()
    cursor.execute('INSERT INTO tasks (title, description, done) VALUES (?, ?, ?)',
                   (title, description, int(done)))
    db.commit()
    new_id = cursor.lastrowid
    db.close()
 
    return jsonify({"id": new_id, "title": title, "description": description, "done": done}), 201

@app.route('/tasks/<int:task_id>', methods=['PUT'])
def update_task(task_id):
    data = request.get_json() or {}
    title = data.get('title')
    description = data.get('description', '')
    done = data.get('done', False)

    if not title:
        return jsonify({'error': 'Title is required'}), 400

    db = get_db()
    cursor = db.cursor()
    cursor.execute(
        'UPDATE tasks SET title = ?, description = ?, done = ? WHERE id = ?',
        (title, description, int(done), task_id),
    )
    db.commit()
    db.close()

    return (
        jsonify({
            'id': task_id,
            'title': title,
            'description': description,
            'done': done,
        }),
        200,
    )

@app.route('/tasks/<int:task_id>', methods=['DELETE'])
def delete_task(task_id):
    db = get_db()
    db.execute('DELETE FROM tasks WHERE id = ?', (task_id,))
    db.commit()
    db.close()

    return jsonify({'message': 'Task deleted successfully'}), 200
# ------------------- Run -------------------
if __name__ == '__main__':
    run_simple("localhost", 5000, app)


 * Running on http://localhost:5000
Press CTRL+C to quit
127.0.0.1 - - [21/Sep/2026 20:35:42] "GET /tasks HTTP/1.1" 200 -
127.0.0.1 - - [21/Sep/2026 20:36:34] "GET /tasks/1 HTTP/1.1" 404 -
127.0.0.1 - - [21/Sep/2026 20:48:06] "POST /tasks HTTP/1.1" 201 -


In [ ]:
pip install flask

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
